# Efficient Frontier

A combined study of two-asset and multi-asset mean–variance portfolio construction. Both sections use the same monthly industry-return dataset and the shared functions in `risk_kit.py`.

**Sample:** January 1996–December 2000. **Frequency:** monthly. Expected returns and covariance are annualised before portfolio optimisation.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import risk_kit as rk

plt.rcParams.update({"figure.figsize": (9, 5.5), "axes.grid": True, "grid.alpha": 0.25})
industries = rk.get_ind_returns()
estimation_returns = industries.loc["1996":"2000"]
er = rk.annualize_rets(estimation_returns, 12)
cov = estimation_returns.cov() * 12
assert not estimation_returns.empty, "The selected estimation window contains no observations."
assert er.index.equals(cov.index), "Expected returns and covariance assets are misaligned."
print(f"Dataset: {industries.shape[1]} industries | {len(industries)} monthly observations | {industries.index.min()} to {industries.index.max()}")

## 1. Two-asset efficient frontier
A two-asset opportunity set shows the return–risk combinations available by varying the allocation between two industries. The curve depends on each asset's expected return, volatility and covariance.

In [ ]:
assets_2 = ["Games", "Fin"]
er_2 = er[assets_2]
cov_2 = cov.loc[assets_2, assets_2]
ax = rk.plot_ef2(101, er_2, cov_2)
ax.set_title("Two-asset opportunity set — Games and Financials")
plt.tight_layout()

![Two-asset efficient frontier](../outputs/two_asset_frontier.svg)

**Interpretation:** Each point is a fully invested long-only mix. The curve shows the risk–return trade-off; diversification can reduce volatility when return correlation is below one.

## 2. N-asset efficient frontier
For each target return, the optimiser minimises portfolio volatility subject to weights summing to one and each weight remaining between zero and one.

In [ ]:
assets_n = ["Smoke", "Fin", "Games", "Coal"]
er_n = er[assets_n]
cov_n = cov.loc[assets_n, assets_n]
target_return = 0.15
weights_15 = rk.minimize_vol(target_return, er_n, cov_n)
assert np.isclose(weights_15.sum(), 1.0, atol=1e-6)
assert (weights_15 >= -1e-8).all()
pd.Series(weights_15, index=assets_n, name=f"Weights at {target_return:.0%} target")

In [ ]:
frontier_weights = rk.optimal_weights(60, er_n, cov_n)
frontier = pd.DataFrame({
    "Return": [rk.portfolio_return(w, er_n) for w in frontier_weights],
    "Volatility": [rk.portfolio_vol(w, cov_n) for w in frontier_weights],
})
ax = rk.plot_ef(60, er_n, cov_n)
ax.set_title("Four-asset long-only efficient frontier")
plt.tight_layout()
frontier.head()

![N-asset efficient frontier](../outputs/n_asset_frontier.svg)

**Interpretation:** Every point is the lowest-volatility allocation for a specified target return. The optimiser uses the full covariance matrix, so diversification effects are not captured by looking at individual asset volatility alone.

## 3. Compare reference portfolios
Equal weighting is a transparent benchmark. The global minimum-variance portfolio minimises volatility without using expected-return estimates. The maximum-Sharpe portfolio and capital market line additionally depend on the assumed risk-free rate.

In [ ]:
risk_free_rate = 0.03  # Illustrative annual assumption
ax = rk.plot_ef(60, er_n, cov_n, riskfree_rate=risk_free_rate,
                show_cml=True, show_ew=True, show_gmv=True)
ax.set_title("Efficient frontier with GMV and maximum-Sharpe portfolios")
plt.tight_layout()

![GMV and capital market line](../outputs/gmv_and_cml.svg)

**Interpretation:** GMV is covariance-driven; maximum Sharpe also uses expected returns and the risk-free assumption. The plotted allocations are estimates from one historical window, not forward-looking recommendations.

## 4. Estimation sensitivity
Maximum-Sharpe allocations can change materially when expected returns change slightly. The covariance matrix below is held fixed to isolate sensitivity to expected-return assumptions.

In [ ]:
pair = ["Food", "Steel"]
pair_cov = cov.loc[pair, pair]
scenarios = {
    "Historical estimates": er[pair].to_numpy(),
    "Scenario A": np.array([0.11, 0.12]),
    "Scenario B": np.array([0.10, 0.13]),
}
pd.DataFrame({name: rk.msr(risk_free_rate, values, pair_cov)
              for name, values in scenarios.items()}, index=pair)

## Key conclusions
- The two-asset curve illustrates diversification geometrically; the N-asset frontier generalises the same idea through constrained optimisation.
- Expected returns and covariance must use consistent annual units.
- Maximum-Sharpe allocations can be unstable when expected-return estimates change.
- All results are in-sample, long-only and exclude transaction costs, taxes and liquidity constraints.


## Reproducibility and execution notes

- Run this notebook from the repository root after installing `requirements.txt`.
- The shared calculations are implemented in `risk_kit.py`; this notebook focuses on the research question and interpretation.
- Code cells are intentionally saved without stale execution outputs. Run **Kernel → Restart Kernel and Run All Cells** to reproduce the analysis from a clean state.
- Model parameters and historical sample choices are illustrative and should be changed only with the resulting assumptions documented.
